# Prepare Shareable PR Dataset

Merge three PR CSV files, remove internal/sensitive columns, save a share-ready CSV, and zip the result.

In [ ]:
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

import pandas as pd

## Inputs

Edit `INPUT_CSVS` if you want to merge a different set of three CSV files.

In [ ]:
INPUT_CSVS = [
    Path("../data/code_complexity_march_month.csv"),
    Path("../data/code_complexity_april_month.csv"),
    Path("../data/code_complexity_may_month.csv"),
]

OUTPUT_DIR = Path("data/share_ready")
OUTPUT_CSV = OUTPUT_DIR / "merged_pr_data_share_ready.csv"
OUTPUT_ZIP = OUTPUT_DIR / "merged_pr_data_share_ready.zip"
OVERWRITE_OUTPUTS = False

COLUMNS_TO_REMOVE = [
    "billable_owner_id",
    "oid",
    "standardized_complexity_score",
    "complexity_score",
]

## Validate And Load

In [ ]:
if len(INPUT_CSVS) != 3:
    raise ValueError(f"Expected exactly 3 CSV paths, got {len(INPUT_CSVS)}")

missing_files = [path for path in INPUT_CSVS if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Missing input CSV files: {missing_files}")

input_frames = []
input_summaries = []

for path in INPUT_CSVS:
    frame = pd.read_csv(path)
    frame["source_file"] = path.name
    input_frames.append(frame)
    input_summaries.append({
        "input_file": str(path),
        "rows": len(frame),
        "columns": frame.shape[1],
    })

input_summary = pd.DataFrame(input_summaries)
input_summary

## Merge And Remove Columns

In [ ]:
merged_df = pd.concat(input_frames, ignore_index=True)

columns_removed = [col for col in COLUMNS_TO_REMOVE if col in merged_df.columns]
columns_missing = [col for col in COLUMNS_TO_REMOVE if col not in merged_df.columns]
share_ready_df = merged_df.drop(columns=columns_removed)

merge_summary = pd.DataFrame([
    {
        "input_rows_total": sum(len(frame) for frame in input_frames),
        "output_rows": len(share_ready_df),
        "input_columns_after_source_file": merged_df.shape[1],
        "output_columns": share_ready_df.shape[1],
        "columns_removed": ", ".join(columns_removed),
        "requested_columns_not_found": ", ".join(columns_missing),
    }
])

merge_summary

In [ ]:
share_ready_df.head()

## Save And Zip

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

existing_outputs = [path for path in [OUTPUT_CSV, OUTPUT_ZIP] if path.exists()]
if existing_outputs and not OVERWRITE_OUTPUTS:
    raise FileExistsError(
        f"Output already exists: {existing_outputs}. Set OVERWRITE_OUTPUTS = True to replace."
    )

share_ready_df.to_csv(OUTPUT_CSV, index=False)

with ZipFile(OUTPUT_ZIP, mode="w", compression=ZIP_DEFLATED) as zip_file:
    zip_file.write(OUTPUT_CSV, arcname=OUTPUT_CSV.name)

print(f"Saved CSV: {OUTPUT_CSV}")
print(f"Saved ZIP: {OUTPUT_ZIP}")
print(f"Rows: {len(share_ready_df):,}")
print(f"Columns: {share_ready_df.shape[1]:,}")

## Final Validation

In [ ]:
remaining_removed_columns = [col for col in COLUMNS_TO_REMOVE if col in share_ready_df.columns]

validation_summary = pd.DataFrame([
    {
        "csv_exists": OUTPUT_CSV.exists(),
        "zip_exists": OUTPUT_ZIP.exists(),
        "zip_size_mb": round(OUTPUT_ZIP.stat().st_size / (1024 * 1024), 2) if OUTPUT_ZIP.exists() else None,
        "removed_columns_still_present": ", ".join(remaining_removed_columns),
        "ready_to_share": OUTPUT_CSV.exists() and OUTPUT_ZIP.exists() and not remaining_removed_columns,
    }
])

validation_summary